# z606 - Features dirigidas (Etapa 7)
Sobre `tb_features_FE604.parquet`: saca las 4 features en 0, agrega indice macro (leave-one-out) y ratio de tendencia corta/larga.

In [1]:
!pip install -q polars pyarrow

In [2]:
import os
import polars as pl
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'FE607',
    'features_path': '/home/ds/datasets/tb_features_FE604.parquet'
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/FE607


In [4]:
df = pl.read_parquet(PARAM['features_path'])
print(df.shape)

(31522, 72)


## 1. Sacar features con importancia 0
Confirmado en feature_importance de LGB03: estas 4 nunca fueron usadas por el modelo (tienen sentido a nivel cliente-producto, no a nivel producto agregado).

In [5]:
columnas_sin_uso = [
    "producto_joven",
    "meses_desde_ultima_compra",
    "racha_actual_sin_compra",
    "mayor_racha_historica",
]
df = df.drop([c for c in columnas_sin_uso if c in df.columns])
print(df.shape)

(31522, 68)


## 2. Indice macro (leave-one-out sobre TODOS los productos)
Mismo criterio que los ratios de grupo de z604 (excluye al propio producto), pero sin agrupar por categoria -- el "grupo" es el universo completo. Le da al modelo la tendencia macro de forma directa, en vez de inferirla via periodo_m.

In [6]:
def ratio_leave_one_out(df, group_cols, metric):
    nombre_grupo = "_".join(group_cols) if group_cols else "macro"
    agg = df.group_by(group_cols + ["periodo"]).agg(
        pl.col(metric).sum().alias("_suma_grupo"),
        pl.len().alias("_n_grupo")
    )
    out = df.join(agg, on=group_cols + ["periodo"], how="left")
    out = out.with_columns(
        (
            (pl.col("_suma_grupo") - pl.col(metric))
            / (pl.col("_n_grupo") - 1).clip(lower_bound=1)
        ).alias(f"{metric}_prom_{nombre_grupo}_excl")
    )
    out = out.with_columns(
        (pl.col(metric) / (pl.col(f"{metric}_prom_{nombre_grupo}_excl") + 1e-6)).alias(
            f"ratio_{metric}_{nombre_grupo}"
        )
    )
    return out.drop(["_suma_grupo", "_n_grupo", f"{metric}_prom_{nombre_grupo}_excl"])

df = ratio_leave_one_out(df, [], "tn")

## 3. Ratio de tendencia: media corta vs media larga
`tn_media_3 / tn_media_12`. Ambas ya son top del ranking por separado; el ratio capta si el producto esta acelerando o frenando respecto a su propio promedio anual, algo que ninguna de las dos por separado mide.

In [7]:
df = df.with_columns(
    (pl.col("tn_media_3") / (pl.col("tn_media_12") + 1e-6)).alias("ratio_tendencia_3_12")
)

## 4. Guardar

In [8]:
salida = os.path.join(ruta, "tb_features_FE607.parquet")
df.write_parquet(salida)
print(salida)
print(df.shape)

/home/ds/exp/FE607/tb_features_FE607.parquet
(31522, 70)
